In [1]:
import mlflow
import os 
from getpass import getpass
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input
from tensorflow.keras.optimizers import RMSprop, SGD, Adam
from tensorflow.keras import regularizers
from keras.callbacks import ModelCheckpoint, EarlyStopping
import mlflow.keras

dataset=mnist.load_data()
(x_train, y_train), (x_test, y_test) = dataset

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

x_train /= 255  # x_trainv = x_trainv/255
x_test /= 255

num_classes=10
y_trainc = keras.utils.to_categorical(y_train, num_classes)
y_testc = keras.utils.to_categorical(y_test, num_classes)

x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)


In [2]:
REPO_NAME= "Curso-de-redes-neuronales-FCFM"
REPO_OWNER= "Oscar-Eduardo-Gonzalez-Jaramillo"  #Escribir nombre de repositorio
USER_NAME = "Oscar-Eduardo-Gonzalez-Jaramillo" #Escribir su usuario

In [3]:
os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = getpass('Enter your DAGsHub access token or password: ')
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow')


In [4]:
model = Sequential()

model.add(Dense(15, activation='relu', input_shape=(784,)))
model.add(Dense(num_classes, activation='softmax'))


c:\Users\Oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 15)             │        11,775 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,935 (46.62 KB)

 Trainable params: 11,935 (46.62 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
print(mlflow.get_tracking_uri())


https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow


In [7]:
from tensorflow.keras.models import clone_model

mlflow.tensorflow.autolog(log_models=True)
mlflow.set_experiment("network_keras")
with mlflow.start_run() as run:

    model = clone_model(model)
    
    earlystop = EarlyStopping(
        monitor='val_loss',
        mode='min',
        restore_best_weights=True,
        patience=30,
        verbose=1
    )
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )
    history = model.fit(
        x_train,
        y_trainc,
        batch_size=200,
        epochs=300,
        verbose=1,
        validation_data=(x_test, y_testc),
        callbacks=[earlystop]
    )

    model_path = "mi_modelo_keras.keras"
    model.save(model_path)
    print(f"Modelo guardado en: {model_path}")
    mlflow.log_artifact(model_path, artifact_path="model")


2025/09/16 23:12:49 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.7.4 <= tensorflow <= 2.19.0, but the installed version is 2.20.0. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


Epoch 1/300
258/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2211 - loss: 2.1846

300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3737 - loss: 1.9443 - val_accuracy: 0.6361 - val_loss: 1.5064
Epoch 2/300
258/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6956 - loss: 1.3683

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7461 - loss: 1.2058 - val_accuracy: 0.7997 - val_loss: 0.9655
Epoch 3/300
258/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8071 - loss: 0.9126

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.8182 - loss: 0.8404 - val_accuracy: 0.8393 - val_loss: 0.7217
Epoch 4/300
278/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8394 - loss: 0.6984

...

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9587 - loss: 0.1433 - val_accuracy: 0.9491 - val_loss: 0.1770
Epoch 202/300
277/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9593 - loss: 0.1416

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9586 - loss: 0.1430 - val_accuracy: 0.9491 - val_loss: 0.1766
Epoch 203/300
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9589 - loss: 0.1427 - val_accuracy: 0.9484 - val_loss: 0.1774
Epoch 204/300
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9585 - loss: 0.1425 - val_accuracy: 0.9490 - val_loss: 0.1769
Epoch 205/300
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9589 - loss: 0.1422 - val_accuracy: 0.9491 - val_loss: 0.1769
Epoch 206/300
299/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9593 - loss: 0.1406

300/300 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.9589 - loss: 0.1420 - val_accuracy: 0.9487 - val_loss: 0.1761
Epoch 207/300
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9589 - loss: 0.1416 - val_accuracy: 0.9488 - val_loss: 0.1761
Epoch 208/300
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9589 - loss: 0.1414 - val_accuracy: 0.9492 - val_loss: 0.1761
Epoch 209/300
257/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9605 - loss: 0.1404

300/300 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.9592 - loss: 0.1412 - val_accuracy: 0.9495 - val_loss: 0.1755
Epoch 210/300
294/300 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9601 - loss: 0.1372🏃 View run rumbling-squid-131 at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/15/runs/2f63bc4635fc494eae146b39eb90285c
🧪 View experiment at: https://dagshub.com/Oscar-Eduardo-Gonzalez-Jaramillo/Curso-de-redes-neuronales-FCFM.mlflow/#/experiments/15


KeyboardInterrupt: 

In [8]:
print("Mejor época según EarlyStopping:", earlystop.best_epoch)
print("Mejor val_loss registrado:", earlystop.best)


Mejor época según EarlyStopping: 299
Mejor val_loss registrado: 0.26189860701560974
